## Example 1: S3 bucket block configuration

In [ ]:
import prefect_aws
from prefect_aws import AwsCredentials
from prefect_aws.s3 import S3Bucket

# Example 1: S3Bucket block configuration. Nothing here contacts AWS.
print("Example 1 - S3 bucket block configuration")

credentials = AwsCredentials(
    aws_access_key_id="demo-key",
    aws_secret_access_key="demo-secret",
    region_name="us-east-1",
)
s3_bucket = S3Bucket(
    bucket_name="demo-prefect-bucket",
    credentials=credentials,
    bucket_folder="reports",
)

print(f"S3 Bucket block created:")
print(f"  Bucket name: {s3_bucket.bucket_name}")
print(f"  Bucket folder: {s3_bucket.bucket_folder}")
print(f"  Region: {s3_bucket.credentials.region_name}")
print(f"  Block type: {type(s3_bucket).__name__}")
assert s3_bucket.bucket_name == "demo-prefect-bucket", s3_bucket.bucket_name
assert s3_bucket.bucket_folder == "reports", s3_bucket.bucket_folder
assert s3_bucket.credentials.region_name == "us-east-1", s3_bucket.credentials.region_name

# The secret is masked when printed and readable on request.
assert str(s3_bucket.credentials.aws_secret_access_key) == "**********"
assert s3_bucket.credentials.aws_secret_access_key.get_secret_value() == "demo-secret"

# Building the S3 client is local: boto3 only contacts AWS when a request is sent.
s3_client = s3_bucket.credentials.get_s3_client()
assert s3_client.meta.region_name == "us-east-1", s3_client.meta.region_name
assert s3_client.meta.service_model.service_name == "s3"

s3_operations = [
    "upload_from_path",
    "download_object_to_path",
    "upload_from_file_object",
    "download_object_to_file_object",
    "upload_from_folder",
    "list_objects",
    "read_path",
    "write_path",
]
missing = [op for op in s3_operations if not hasattr(s3_bucket, op)]
assert not missing, f"S3Bucket is missing {missing}"
print(f"\nS3 operations present: {', '.join(s3_operations)}")

print("Example 1 completed successfully")


## Example 2: AWS credentials and boto3 clients

In [ ]:
from prefect_aws import AwsCredentials, MinIOCredentials
from prefect import flow

# Example 2: AWS credentials and boto3 clients. Sessions and clients are built locally.
print("Example 2 - AWS credentials and boto3 clients")

aws_creds = AwsCredentials(
    aws_access_key_id="demo-access-key-id",
    aws_secret_access_key="demo-secret-access-key",
    region_name="us-east-1"
)

print(f"AWS Credentials configured:")
print(f"  Region: {aws_creds.region_name}")
print(f"  Credentials type: {type(aws_creds).__name__}")

client_methods = [
    "get_boto3_session",
    "get_client",
    "get_s3_client",
    "get_secrets_manager_client"
]
for method in client_methods:
    assert hasattr(aws_creds, method), method

session = aws_creds.get_boto3_session()
assert session.region_name == "us-east-1", session.region_name
session_creds = session.get_credentials()
assert session_creds.access_key == "demo-access-key-id", session_creds.access_key

# MinIO credentials (S3-compatible storage)
minio_creds = MinIOCredentials(
    minio_root_user="minioadmin",
    minio_root_password="minioadmin",
    region_name="us-east-1"
)
print(f"\nMinIO Credentials (S3-compatible):")
print(f"  Region: {minio_creds.region_name}")
print(f"  Root user: {minio_creds.minio_root_user}")
assert minio_creds.minio_root_user == "minioadmin"
assert str(minio_creds.minio_root_password) == "**********"
assert minio_creds.minio_root_password.get_secret_value() == "minioadmin"

# A flow that builds one client per service from the same credentials block
@flow(name="aws-service-access")
def aws_service_access_flow():
    s3 = aws_creds.get_s3_client()
    secrets = aws_creds.get_secrets_manager_client()
    lambda_client = aws_creds.get_client("lambda")
    return {
        client.meta.service_model.service_name: client.meta.region_name
        for client in (s3, secrets, lambda_client)
    }

result = aws_service_access_flow()
print(f"\nFlow result (service -> region): {result}")
assert result == {"s3": "us-east-1", "secretsmanager": "us-east-1", "lambda": "us-east-1"}, result

print("Example 2 completed successfully")


## Example 3: Credentials and Secrets Manager blocks

In [ ]:
from prefect_aws import AwsCredentials
from prefect_aws.secrets_manager import AwsSecret

# Example 3: Credentials and Secrets Manager blocks
print("Example 3 - Credentials and Secrets Manager blocks")

aws_creds = AwsCredentials(
    aws_access_key_id="demo-access-key",
    aws_secret_access_key="demo-secret-key",
    region_name="us-west-2"
)

print(f"AWS Credentials configured:")
print(f"  Region: {aws_creds.region_name}")
assert aws_creds.aws_access_key_id == "demo-access-key"
assert aws_creds.aws_secret_access_key.get_secret_value() == "demo-secret-key"

cred_methods = [
    "get_boto3_session",
    "get_client",
    "get_s3_client",
    "get_secrets_manager_client"
]
missing = [m for m in cred_methods if not hasattr(aws_creds, m)]
assert not missing, f"AwsCredentials is missing {missing}"

session = aws_creds.get_boto3_session()
print(f"  boto3 session region: {session.region_name}")
assert session.region_name == "us-west-2", session.region_name

secrets_client = aws_creds.get_secrets_manager_client()
assert secrets_client.meta.service_model.service_name == "secretsmanager"
assert secrets_client.meta.region_name == "us-west-2", secrets_client.meta.region_name

# AWS Secrets Manager block
aws_secret = AwsSecret(
    secret_name="demo/app/config",
    aws_credentials=aws_creds
)

print(f"\nAWS Secrets Manager block:")
print(f"  Secret name: {aws_secret.secret_name}")
assert aws_secret.secret_name == "demo/app/config"
assert aws_secret.aws_credentials.region_name == "us-west-2"
for method in ("read_secret", "write_secret", "delete_secret"):
    assert hasattr(aws_secret, method), method

print("Example 3 completed successfully")


## Example 4: Lambda function block in a flow

In [ ]:
from prefect_aws import AwsCredentials
from prefect_aws.lambda_function import LambdaFunction
from prefect import flow
import json

# Example 4: Lambda function block used inside a flow (no function is invoked)
print("Example 4 - Lambda function block in a flow")

lambda_func = LambdaFunction(
    function_name="demo-data-processor",
    qualifier="live",
    aws_credentials=AwsCredentials(
        aws_access_key_id="demo-key",
        aws_secret_access_key="demo-secret",
        region_name="us-east-1",
    ),
)

print(f"Lambda Function configured:")
print(f"  Function name: {lambda_func.function_name}")
print(f"  Qualifier: {lambda_func.qualifier}")
assert lambda_func.function_name == "demo-data-processor", lambda_func.function_name
assert lambda_func.qualifier == "live", lambda_func.qualifier
assert hasattr(lambda_func, "invoke")

sample_payload = {
    "action": "process_data",
    "input_s3_bucket": "data-input-bucket",
    "output_s3_bucket": "data-output-bucket",
    "parameters": {
        "batch_size": 1000,
        "processing_mode": "standard"
    }
}

@flow(name="lambda-data-processing")
def lambda_processing_flow(payload: dict) -> dict:
    """Prepare a Lambda request: the client and the JSON body invoke() would send"""
    client = lambda_func.aws_credentials.get_client("lambda")
    body = json.dumps(payload).encode()
    return {
        "function_name": lambda_func.function_name,
        "service": client.meta.service_model.service_name,
        "region": client.meta.region_name,
        "payload": json.loads(body),
    }

result = lambda_processing_flow(sample_payload)
print(f"\nFlow result: function={result['function_name']}, service={result['service']}, region={result['region']}")
assert result == {
    "function_name": "demo-data-processor",
    "service": "lambda",
    "region": "us-east-1",
    "payload": sample_payload,
}, result

print("Example 4 completed successfully")


## Example 5: Data pipeline flow with S3 and Lambda blocks

In [ ]:
from prefect import flow, task, get_run_logger
from prefect_aws.s3 import S3Bucket
from prefect_aws.lambda_function import LambdaFunction
from prefect_aws import AwsCredentials

# Example 5: A pipeline flow wired to S3 and Lambda blocks. The tasks only log and pass
# results along; no AWS request is made.
print("Example 5 - Data pipeline flow")

aws_credentials = AwsCredentials(
    aws_access_key_id="demo-key",
    aws_secret_access_key="demo-secret",
    region_name="us-east-1"
)

input_bucket = S3Bucket(
    bucket_name="data-pipeline-input",
    credentials=aws_credentials
)

output_bucket = S3Bucket(
    bucket_name="data-pipeline-output",
    credentials=aws_credentials
)

processor_lambda = LambdaFunction(
    function_name="data-processor",
    aws_credentials=aws_credentials
)

print("AWS Pipeline Components:")
print(f"  Input bucket: {input_bucket.bucket_name}")
print(f"  Output bucket: {output_bucket.bucket_name}")
print(f"  Processor: {processor_lambda.function_name}")
assert processor_lambda.aws_credentials.region_name == "us-east-1"

@task(name="validate-input")
def validate_input_data():
    """Validate input data exists and is properly formatted"""
    logger = get_run_logger()
    logger.info("Validating input data in %s", input_bucket.bucket_name)
    return {"status": "valid", "file_count": 5}

@task(name="process-data")
def process_data_batch(validation_result):
    """Process the validated files"""
    logger = get_run_logger()
    logger.info("Processing %d files with %s", validation_result["file_count"], processor_lambda.function_name)
    return {"processed": validation_result["file_count"], "status": "complete"}

@task(name="upload-results")
def upload_results(processing_result):
    """Report where the processed results go"""
    logger = get_run_logger()
    logger.info("Uploading %d processed files", processing_result["processed"])
    return {"uploaded": processing_result["processed"], "location": f"s3://{output_bucket.bucket_name}/"}

@flow(name="aws-data-pipeline")
def aws_data_pipeline():
    """Three tasks run in order, each taking the previous result"""
    validation = validate_input_data()
    processing = process_data_batch(validation)
    return upload_results(processing)

print(f"\nPipeline Flow: {aws_data_pipeline.name}")
result = aws_data_pipeline()
print(f"Pipeline result: {result}")
assert result == {"uploaded": 5, "location": "s3://data-pipeline-output/"}, result

print("Example 5 completed successfully")
